# 06 CNN基础与图像分类

## 什么是卷积神经网络（CNN）？

卷积神经网络（Convolutional Neural Network, CNN）是深度学习在计算机视觉领域最重要的架构之一。与全连接网络不同，CNN通过**局部连接**和**权重共享**两大特性，极大地减少了参数量，同时保留了对图像空间结构的感知能力。

### CNN的核心组件

1. **卷积层（Convolutional Layer）**：通过可学习的卷积核在输入图像上滑动，提取局部特征（边缘、纹理、形状等）
2. **池化层（Pooling Layer）**：通过下采样减小特征图尺寸，提供平移不变性，降低计算量
3. **激活函数（Activation Function）**：引入非线性，使网络能学习复杂模式
4. **全连接层（Fully Connected Layer）**：将提取的特征映射到最终的分类输出

### 图像分类任务

图像分类是计算机视觉的基础任务：给定一张输入图像，判断它属于哪个预定义类别。MNIST手写数字识别是最经典的入门案例。

本教程将带你从零开始：手写二维卷积 → 构建CNN → 训练MNIST分类器 → 可视化模型内部。

In [ ]:
# ===== 环境准备与导入 =====
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
from torchvision import datasets

from scipy import signal
from sklearn.metrics import confusion_matrix, classification_report
import time
from pathlib import Path

# 字体设置
mpl.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
mpl.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

# 设备选择
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"PyTorch {torch.__version__}  |  torchvision {torchvision.__version__}")
print(f"NumPy {np.__version__}")
print(f"计算设备: {DEVICE}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")


## 手动实现二维卷积

在调用PyTorch的 `Conv2d` 之前，我们先亲手实现一个卷积操作，理解其底层计算过程。

### 卷积运算公式

对于输入矩阵 $I$ 和卷积核 $K$，输出在位置 $(i,j)$ 的值为：

$$\text{output}(i, j) = \sum_{u=0}^{k_h-1} \sum_{v=0}^{k_w-1} I(i+u,\ j+v) \cdot K(u, v)$$

参数说明：
- `padding=0`：不在输入四周填充像素
- `stride=1`：卷积核每次滑动1个像素
- 输出尺寸：$H_{out} = H_{in} - K_h + 1$，$W_{out} = W_{in} - K_w + 1$

下面的代码使用NumPy嵌套循环实现这一计算过程。

In [ ]:
# ===== 手动实现二维卷积（numpy嵌套循环）=====

def conv2d_manual(input_matrix, kernel, padding=0, stride=1):
    '''
    手动实现二维卷积（互相关操作，深度学习中的"卷积"实际上是互相关）。

    参数:
        input_matrix:  输入矩阵 (H, W) numpy数组
        kernel:        卷积核 (kH, kW) numpy数组
        padding:       填充像素数
        stride:        步长

    返回:
        output: 卷积输出矩阵
    '''
    # 添加padding
    if padding > 0:
        input_padded = np.pad(input_matrix, padding, mode='constant')
    else:
        input_padded = input_matrix

    H_in, W_in = input_padded.shape
    kH, kW = kernel.shape

    # 计算输出尺寸
    H_out = (H_in - kH) // stride + 1
    W_out = (W_in - kW) // stride + 1

    # 初始化输出矩阵
    output = np.zeros((H_out, W_out))

    # 嵌套循环执行卷积
    for i in range(H_out):
        for j in range(W_out):
            # 取出当前窗口
            window = input_padded[i*stride : i*stride + kH,
                                  j*stride : j*stride + kW]
            # 逐元素乘积求和
            output[i, j] = np.sum(window * kernel)

    return output


# ---- 测试：在示例矩阵上应用边缘检测核 ----
# 创建测试图像（中心为白色方块的黑色背景）
test_img = np.zeros((8, 8))
test_img[2:6, 2:6] = 1.0

# 水平边缘检测核（Sobel X方向）
edge_kernel = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1]
], dtype=np.float64)

# 垂直边缘检测核（Sobel Y方向）
edge_kernel_y = np.array([
    [-1, -2, -1],
    [ 0,  0,  0],
    [ 1,  2,  1]
], dtype=np.float64)

# 执行手动卷积
result_x = conv2d_manual(test_img, edge_kernel, padding=0, stride=1)
result_y = conv2d_manual(test_img, edge_kernel_y, padding=0, stride=1)
result_combined = np.sqrt(result_x**2 + result_y**2)

# 可视化
fig, axes = plt.subplots(1, 5, figsize=(16, 4))

axes[0].imshow(test_img, cmap='gray')
axes[0].set_title('原始图像 (8x8)', fontsize=12)
axes[0].axis('off')

axes[1].imshow(edge_kernel, cmap='RdBu', vmin=-2, vmax=2)
axes[1].set_title('Sobel-X 卷积核 (3x3)', fontsize=12)
axes[1].axis('off')

axes[2].imshow(result_x, cmap='gray')
axes[2].set_title(f'水平边缘响应\n输出尺寸: {result_x.shape}', fontsize=12)
axes[2].axis('off')

axes[3].imshow(result_y, cmap='gray')
axes[3].set_title(f'垂直边缘响应\n输出尺寸: {result_y.shape}', fontsize=12)
axes[3].axis('off')

axes[4].imshow(result_combined, cmap='gray')
axes[4].set_title(f'综合边缘强度\n输出尺寸: {result_combined.shape}', fontsize=12)
axes[4].axis('off')

plt.suptitle('手动二维卷积 — 边缘检测演示', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("观察:")
print("  1. 输入 8×8，卷积核 3×3，输出变为 6×6（尺寸缩减）")
print("  2. Sobel-X 检测垂直边缘（左右亮度差异）")
print("  3. Sobel-Y 检测水平边缘（上下亮度差异）")
print(f"  4. 综合边缘强度范围: [{result_combined.min():.1f}, {result_combined.max():.1f}]")


In [ ]:
# ===== 验证：手动卷积 vs scipy.signal.convolve2d =====

from scipy import signal

# 用scipy的convolve2d执行同样的操作
result_scipy = signal.convolve2d(test_img, edge_kernel, mode='valid')

# 比较结果
difference = np.abs(result_x - result_scipy)

print("手动卷积 vs scipy.signal.convolve2d 对比:")
print(f"  最大误差: {difference.max():.2e}")
print(f"  平均误差: {difference.mean():.2e}")
print(f"  完全匹配: {np.allclose(result_x, result_scipy, atol=1e-10)}")

# 详细输出两个结果的对比
print(f"\n手动卷积结果:\n{result_x}")
print(f"\nscipy卷积结果:\n{result_scipy}")

# 可视化对比
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

axes[0].imshow(result_x, cmap='gray')
axes[0].set_title('手动卷积结果', fontsize=12)
for i in range(result_x.shape[0]):
    for j in range(result_x.shape[1]):
        axes[0].text(j, i, f'{result_x[i,j]:.0f}', ha='center', va='center', fontsize=8)

axes[1].imshow(result_scipy, cmap='gray')
axes[1].set_title('scipy.signal.convolve2d 结果', fontsize=12)
for i in range(result_scipy.shape[0]):
    for j in range(result_scipy.shape[1]):
        axes[1].text(j, i, f'{result_scipy[i,j]:.0f}', ha='center', va='center', fontsize=8)

axes[2].imshow(difference, cmap='hot')
axes[2].set_title(f'差异图 (max={difference.max():.1e})', fontsize=12)
axes[2].axis('off')

# 同时测试不同padding模式
result_same = signal.convolve2d(test_img, edge_kernel, mode='same')
axes[3].imshow(result_same, cmap='gray')
axes[3].set_title(f'scipy mode="same"\n输出尺寸: {result_same.shape}', fontsize=12)
axes[3].axis('off')

plt.suptitle('手动卷积验证', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("结论: 手动实现与scipy结果完全一致，验证了卷积运算的正确性。")


## 使用PyTorch构建CNN

理解了卷积的底层原理后，我们用PyTorch搭建一个完整的CNN模型。模型架构为经典的类LeNet结构：

```
Conv2d(1, 16, 3)  ->  ReLU  ->  MaxPool2d(2)  ->
Conv2d(16, 32, 3) ->  ReLU  ->  MaxPool2d(2)  ->
Flatten           ->  Linear(32*5*5, 128) -> ReLU ->
Linear(128, 10)   ->  Softmax
```

注意：最后一个卷积层之后特征图的尺寸需要计算。对于MNIST的28x28输入：
- 第一层Conv(3x3)后：28 - 3 + 1 = 26 → Pool(2)后：13
- 第二层Conv(3x3)后：13 - 3 + 1 = 11 → Pool(2)后：5
- Flatten后：32通道 × 5 × 5 = 800维

In [ ]:
# ===== 构建SimpleCNN模型 =====

class SimpleCNN(nn.Module):
    '''简单的CNN模型，用于MNIST手写数字分类'''

    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()

        # 第一个卷积块
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=0, stride=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        # 第二个卷积块
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=0, stride=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # 分类器
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(32 * 5 * 5, 128)  # 32通道 × 5×5 特征图
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, num_classes)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.flatten(x)
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)
        return x


# 实例化模型并打印结构
model = SimpleCNN(num_classes=10).to(DEVICE)
print(model)
print()

# 统计参数
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"总参数量: {total_params:,}")
print(f"可训练参数量: {trainable_params:,}")

# 使用torchsummary风格展示各层输出维度
print("\n层间维度变化:")
print(f"{'层名称':<25} {'输出维度':<20} {'参数量':<12}")
print("-" * 57)

# 模拟输入
x = torch.randn(1, 1, 28, 28).to(DEVICE)
layer_info = [
    ("Input", x.shape, 0),
    ("conv1 (1->16, 3x3)", None, None),
    ("ReLU", None, None),
    ("MaxPool2d(2)", None, None),
    ("conv2 (16->32, 3x3)", None, None),
    ("ReLU", None, None),
    ("MaxPool2d(2)", None, None),
    ("Flatten", None, None),
    ("fc1 (800->128)", None, None),
    ("ReLU", None, None),
    ("fc2 (128->10)", None, None),
]

# 逐步前向传播跟踪
with torch.no_grad():
    out = model.conv1(x)
    print(f"{'conv1 (1->16, 3x3)':<25} {str(list(out.shape)):<20} {16*1*3*3+16:<12}")
    out = model.relu1(out)
    print(f"{'ReLU':<25} {str(list(out.shape)):<20} {0:<12}")
    out = model.pool1(out)
    print(f"{'MaxPool2d(2)':<25} {str(list(out.shape)):<20} {0:<12}")
    out = model.conv2(out)
    print(f"{'conv2 (16->32, 3x3)':<25} {str(list(out.shape)):<20} {32*16*3*3+32:<12}")
    out = model.relu2(out)
    print(f"{'ReLU':<25} {str(list(out.shape)):<20} {0:<12}")
    out = model.pool2(out)
    print(f"{'MaxPool2d(2)':<25} {str(list(out.shape)):<20} {0:<12}")
    out = model.flatten(out)
    print(f"{'Flatten':<25} {str(list(out.shape)):<20} {0:<12}")
    out = model.fc1(out)
    print(f"{'fc1 (800->128)':<25} {str(list(out.shape)):<20} {800*128+128:<12}")
    out = model.relu3(out)
    out = model.fc2(out)
    print(f"{'fc2 (128->10)':<25} {str(list(out.shape)):<20} {128*10+10:<12}")


## 加载MNIST数据集

MNIST（Modified National Institute of Standards and Technology）是手写数字识别领域最经典的数据集，包含60,000张训练图片和10,000张测试图片，每张是28×28的灰度图像，标签为0-9的数字。

torchvision提供了便捷的API来加载MNIST数据集，同时支持自动下载和数据预处理。

In [ ]:
# ===== 加载MNIST数据集 =====

BATCH_SIZE = 64

# 数据预处理：转Tensor + 归一化
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST全局均值和标准差
])

# 加载训练集和测试集
train_dataset = datasets.MNIST(
    root='./data', train=True, download=True, transform=transform
)
test_dataset = datasets.MNIST(
    root='./data', train=False, download=True, transform=transform
)

# 创建DataLoader
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"训练集: {len(train_dataset):,} 张 | 测试集: {len(test_dataset):,} 张")
print(f"类别数: 10 (数字0-9) | 图片尺寸: 28x28 灰度")
print(f"训练batch数: {len(train_loader)} | 测试batch数: {len(test_loader)}")

# 可视化样本batch
examples = iter(train_loader)
example_imgs, example_labels = next(examples)

fig, axes = plt.subplots(4, 8, figsize=(14, 8))
for i, ax in enumerate(axes.flat):
    img = example_imgs[i][0].numpy()
    ax.imshow(img, cmap='gray')
    ax.set_title(f'标签: {example_labels[i].item()}', fontsize=10)
    ax.axis('off')

fig.suptitle('MNIST手写数字 — 训练集样本', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("观察: 同一个数字有各种不同的书写风格——这正是CNN需要学习的"不变性"。")

# 统计类别分布
labels_count = np.bincount(example_labels.numpy(), minlength=10)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(10), labels_count, color='steelblue', edgecolor='white')
ax.set_xlabel('数字类别')
ax.set_ylabel('样本数')
ax.set_title('MNIST训练集类别分布 (一个batch)')
ax.set_xticks(range(10))
plt.show()


## 训练循环

训练CNN的标准流程：

```
for each epoch:
    for each batch:
        1. optimizer.zero_grad()      # 清零梯度
        2. outputs = model(inputs)    # 前向传播
        3. loss = criterion(...)      # 计算损失
        4. loss.backward()            # 反向传播
        5. optimizer.step()           # 更新参数
```

关键概念：
- **Epoch**：完整遍历一次训练集
- **Batch Size**：每次送入模型的样本数
- **Learning Rate**：参数更新的步长
- **CrossEntropyLoss**：内嵌Softmax的多分类损失

In [ ]:
# ===== 训练函数定义 =====

def train_one_epoch(model, loader, criterion, optimizer, device):
    '''训练一个epoch，返回平均损失和准确率'''
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)

        # 标准训练5步
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # 统计
        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = 100.0 * correct / total
    return epoch_loss, epoch_acc


def evaluate(model, loader, criterion, device):
    '''评估模型，返回损失、准确率和预测结果'''
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / total
    epoch_acc = 100.0 * correct / total
    return epoch_loss, epoch_acc, all_preds, all_labels


# ===== 开始训练 =====
model = SimpleCNN(num_classes=10).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

NUM_EPOCHS = 3

history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

print(f"开始训练 SimpleCNN on {DEVICE}")
print(f"Epochs: {NUM_EPOCHS} | Batch Size: {BATCH_SIZE} | Optimizer: Adam(lr=0.001)")
print("-" * 65)
start_time = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    test_loss, test_acc, _, _ = evaluate(model, test_loader, criterion, DEVICE)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)

    print(f"Epoch {epoch:2d}/{NUM_EPOCHS} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | "
          f"Test Loss: {test_loss:.4f} Acc: {test_acc:.2f}%")

total_time = time.time() - start_time
print(f"\n训练完成！总耗时: {total_time:.1f}秒")
print(f"最佳测试准确率: {max(history['test_acc']):.2f}%")

# 绘制训练曲线
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, NUM_EPOCHS + 1)

axes[0].plot(epochs_range, history['train_loss'], 'b-o', label='训练损失', markersize=6)
axes[0].plot(epochs_range, history['test_loss'], 'r-s', label='测试损失', markersize=6)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('损失曲线')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history['train_acc'], 'b-o', label='训练准确率', markersize=6)
axes[1].plot(epochs_range, history['test_acc'], 'r-s', label='测试准确率', markersize=6)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('准确率曲线')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('SimpleCNN — MNIST训练过程', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# ===== 测试集评估 =====

test_loss, test_acc, all_preds, all_labels = evaluate(
    model, test_loader, criterion, DEVICE
)

print(f"最终测试结果:")
print(f"  测试损失: {test_loss:.4f}")
print(f"  测试准确率: {test_acc:.2f}%")
print(f"  错误数: {sum(1 for p, l in zip(all_preds, all_labels) if p != l)} / {len(all_labels)}")

# 使用sklearn的分类报告
from sklearn.metrics import classification_report
print(f"\n分类报告:")
print(classification_report(all_labels, all_preds, digits=4,
                             target_names=[str(i) for i in range(10)]))


## 可视化学习到的卷积核

CNN的第一个卷积层学习到的滤波器是最直观可理解的——它们通常捕捉简单的视觉模式，如边缘、线段、角点等。将16个3×3卷积核可视化为小图像，可以直观地看到模型学到了什么。

In [ ]:
# ===== 可视化第一层卷积核 =====

# 提取conv1的权重
conv1_weights = model.conv1.weight.data.cpu().numpy()  # Shape: (16, 1, 3, 3)
print(f"第一层卷积核权重的形状: {conv1_weights.shape}")
print(f"  16个卷积核，每个1通道，3x3大小")

# 可视化16个滤波器
fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    if i < 16:
        kernel = conv1_weights[i, 0, :, :]  # 取出第i个卷积核
        # 归一化到 [0,1] 以便可视化
        kernel_norm = (kernel - kernel.min()) / (kernel.max() - kernel.min() + 1e-8)
        ax.imshow(kernel_norm, cmap='viridis')
        ax.set_title(f'Filter {i+1}', fontsize=9)
    ax.axis('off')

plt.suptitle('第一层卷积核可视化 (Conv1: 1→16, 3×3)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("观察:")
print("  - 不同的卷积核学习到不同的模式（边缘方向、纹理等）")
print("  - 浅色的正值表示该位置对激活有正贡献")
print("  - 深色的负值表示抑制")

# 同时用RdBu colormap展示原始权重（带正负）
fig, axes = plt.subplots(4, 4, figsize=(8, 8))
vmax = np.max(np.abs(conv1_weights))
for i, ax in enumerate(axes.flat):
    if i < 16:
        kernel = conv1_weights[i, 0, :, :]
        im = ax.imshow(kernel, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
        ax.set_title(f'Filter {i+1}', fontsize=9)
    ax.axis('off')

plt.suptitle('第一层卷积核 (原始权重, RdBu色图)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


## 特征图可视化

特征图（Feature Map）是卷积层对输入图像的响应。将一张真实图片输入网络，观察第一层卷积层的输出，可以直观地看到每个滤波器对输入图像的不同响应模式——哪些滤波器对边缘敏感，哪些对纹理敏感，等等。

In [ ]:
# ===== 特征图可视化 =====

# 取一张测试图片
sample_img, sample_label = test_dataset[0]
sample_input = sample_img.unsqueeze(0).to(DEVICE)  # 增加batch维度

# 获取第一层卷积的输出（在ReLU之前）
with torch.no_grad():
    conv1_out = model.conv1(sample_input)  # Shape: (1, 16, 26, 26)

conv1_out_np = conv1_out.cpu().numpy()[0]  # 去除batch维度

# 显示原图和16个特征图
fig, axes = plt.subplots(4, 5, figsize=(12, 10))

# 第一行第一列显示原图
axes[0, 0].imshow(sample_img[0].numpy(), cmap='gray')
axes[0, 0].set_title(f'输入图像\n标签: {sample_label}', fontsize=9)
axes[0, 0].axis('off')

# 显示16个特征图
for i in range(16):
    row = (i + 1) // 5
    col = (i + 1) % 5
    if row < 4:
        axes[row, col].imshow(conv1_out_np[i], cmap='viridis')
        axes[row, col].set_title(f'Filter {i+1}', fontsize=8)
        axes[row, col].axis('off')

# 隐藏多余的子图
for j in range(17, 20):
    row = j // 5
    col = j % 5
    axes[row, col].axis('off')

plt.suptitle('第一层卷积特征图可视化 (Conv1输出: 16通道 × 26×26)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("观察:")
print("  - 不同的滤波器对不同的笔画方向/边缘有响应")
print("  - 有些滤波器对水平笔画敏感，有些对弯曲笔画敏感")
print("  - 这展示了CNN如何将原始像素分解为多种底层特征")


In [ ]:
# ===== 混淆矩阵 =====

# 收集测试集所有预测结果
test_loss, test_acc, all_preds, all_labels = evaluate(
    model, test_loader, criterion, DEVICE
)

# 计算混淆矩阵
cm = confusion_matrix(all_labels, all_preds)

# 可视化
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax, shrink=0.8)

# 标注数值
thresh = cm.max() / 2
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, format(cm[i, j], 'd') if cm[i, j] > 0 else '',
                ha='center', va='center',
                color='white' if cm[i, j] > thresh else 'black',
                fontsize=10)

ax.set_xlabel('预测类别', fontsize=12)
ax.set_ylabel('真实类别', fontsize=12)
ax.set_title(f'混淆矩阵 (准确率: {test_acc:.2f}%)', fontsize=14)
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xticklabels([str(i) for i in range(10)])
ax.set_yticklabels([str(i) for i in range(10)])

plt.tight_layout()
plt.show()

# 找出最容易被混淆的数字对
errors = []
for i in range(10):
    for j in range(10):
        if i != j:
            errors.append((i, j, cm[i, j]))
errors.sort(key=lambda x: x[2], reverse=True)

print("最容易被混淆的数字对（Top 5）:")
for true_label, pred_label, count in errors[:5]:
    if count > 0:
        print(f"  真实 {true_label} → 预测为 {pred_label} : {count} 次")


## 练习与实践

完成以下练习来巩固学习效果。

In [ ]:
# TODO: 练习1 - 修改CNN架构
# 尝试增加或减少卷积层数量，观察对训练速度和准确率的影响

class CustomCNN(nn.Module):
    '''自定义CNN架构 — 请修改此模型'''
    def __init__(self, num_classes=10):
        super(CustomCNN, self).__init__()

        # === 在此处修改网络结构 ===
        # 尝试:
        #   - 添加第三个卷积层 (如 Conv2d(32, 64, 3))
        #   - 修改卷积核大小 (如 kernel_size=5)
        #   - 添加 Dropout 层防止过拟合
        #   - 修改全连接层大小

        self.conv1 = nn.Conv2d(1, 16, 3)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2)

        self.conv2 = nn.Conv2d(16, 32, 3)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2)

        # TODO: 在此添加更多层
        # self.conv3 = nn.Conv2d(...)
        # self.dropout = nn.Dropout(0.25)

        # 注意：修改后需要重新计算Flatten后的维度
        self.fc1 = nn.Linear(32 * 5 * 5, 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, num_classes)
        # ===========================

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        # TODO: 前向传播中也需要添加新层
        x = x.contiguous().view(x.size(0), -1)
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)
        return x

print("请修改 CustomCNN 类，然后训练并对比与原SimpleCNN的性能差异。")
print("思考题:")
print("  1. 增加层数是否总是提升准确率？")
print("  2. 参数量增加对训练时间的影响有多大？")
print("  3. 什么时候会出现过拟合？如何判断？")


In [ ]:
# TODO: 练习2 - 训练Fashion-MNIST数据集
# 比较与MNIST的分类难度差异

# Fashion-MNIST包含10类时尚商品（T恤、裤子、套衫、裙子、外套、凉鞋、衬衫、运动鞋、包、短靴）
# 图片尺寸和数量与MNIST完全相同，但分类难度更大

# 加载Fashion-MNIST
fashion_train = datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=transform
)
fashion_test = datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=transform
)

fashion_classes = ['T恤/上衣', '裤子', '套衫', '裙子', '外套',
                   '凉鞋', '衬衫', '运动鞋', '包', '短靴']

# 展示Fashion-MNIST样本
examples = iter(DataLoader(fashion_train, batch_size=8, shuffle=True))
imgs, labels = next(examples)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(imgs[i][0].numpy(), cmap='gray')
    ax.set_title(f'{fashion_classes[labels[i]]}', fontsize=10)
    ax.axis('off')
plt.suptitle('Fashion-MNIST 样本', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("思考题:")
print("  1. Fashion-MNIST比MNIST更难分类吗？为什么？")
print("  2. 同样的SimpleCNN在Fashion-MNIST上的准确率是多少？")
print("  3. 哪些类别最容易被混淆？（例如：衬衫 vs T恤）")
print("\n请复制上面的训练代码，训练Fashion-MNIST并分析结果。")


## 实战案例：手写数字识别应用

将训练好的模型应用于实际场景。下面演示如何加载模型、对新样本进行预测，以及如何分析预测结果（包括发现和展示错误分类的案例）。

In [ ]:
# ===== 实战：手写数字识别应用演示 =====

model.eval()

# 从测试集随机抽取10个样本进行演示
import random
indices = random.sample(range(len(test_dataset)), 10)

fig, axes = plt.subplots(2, 5, figsize=(14, 6))

correct_count = 0
for idx, ax in enumerate(axes.flat):
    img, true_label = test_dataset[idx]
    img_tensor = img.unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        output = model(img_tensor)
        probs = torch.softmax(output, dim=1)
        pred_label = output.argmax(dim=1).item()
        confidence = probs[0, pred_label].item()

    is_correct = (pred_label == true_label)
    if is_correct:
        correct_count += 1

    ax.imshow(img[0].numpy(), cmap='gray')
    color = 'green' if is_correct else 'red'
    ax.set_title(f'真实: {true_label} | 预测: {pred_label}\n置信度: {confidence:.2%}',
                 fontsize=9, color=color)
    ax.axis('off')

fig.suptitle(f'手写数字识别演示 (正确率: {correct_count}/10)',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# 找出并展示错误分类的案例
print("\n错误分类案例分析:")
error_samples = []
with torch.no_grad():
    for idx in range(len(test_dataset)):
        if len(error_samples) >= 8:
            break
        img, true_label = test_dataset[idx]
        img_tensor = img.unsqueeze(0).to(DEVICE)
        output = model(img_tensor)
        pred = output.argmax(dim=1).item()
        if pred != true_label:
            probs = torch.softmax(output, dim=1)
            confidences = probs[0].cpu().numpy()
            top2 = np.argsort(confidences)[-2:][::-1]
            error_samples.append((img, true_label, pred, top2, confidences))

if error_samples:
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    for i, (img, true_l, pred_l, top2, confs) in enumerate(error_samples):
        ax = axes[i//4, i%4]
        ax.imshow(img[0].numpy(), cmap='gray')
        ax.set_title(f'真实: {true_l} → 预测: {pred_l}\n'
                     f'Top-2: {top2[0]}({confs[top2[0]]:.2%}), '
                     f'{top2[1]}({confs[top2[1]]:.2%})',
                     fontsize=8, color='red')
        ax.axis('off')
    fig.suptitle('错误分类案例 — 模型为什么出错？', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

print("讨论:")
print("  - 错误通常发生在书写潦草或数字形状模糊的情况下")
print("  - 查看Top-2预测结果，很多时候正确答案在第二选择")
print("  - 这就是为什么实际应用中常需要置信度阈值和人工审核机制")
